# Scikit-learn: Anomaly Detection in Sensor Data

## The Core Question

**How do you detect faults in turbomachinery or structural sensors *before* they become failures?**

You have 5 sensors on a turbine: temperature, vibration, pressure, flow, RPM. You have *thousands* of hours of healthy data. You have *zero* labeled examples of the specific failure you're afraid of. Supervised learning is off the table — you can't train a classifier on failures you've never seen.

**Anomaly detection** is the answer: learn what "normal" looks like, and flag anything that doesn't fit.

## Why Scikit-learn?

- **Isolation Forest** finds anomalies by *isolating* them — anomalies are few and different, so random splits isolate them in fewer steps. No distance calculations, no density grids, scales to high dimensions.
- **DBSCAN** finds anomalies by *clustering* — points that don't belong to any dense cluster are noise. It discovers the cluster structure without telling it how many clusters exist.
- **One consistent API** — `fit`, `predict`, `score_samples` — so you can swap methods and compare them fairly.

## The Workflow

```
 1. COLLECT SENSORS            2. LEARN "NORMAL"              3. FLAG ANOMALIES
 ┌─────────────────────┐      ┌─────────────────────┐        ┌─────────────────────┐
 │ 5 sensors, 1000 h   │      │ Isolation Forest    │        │ score each new      │
 │ (temp, vib, press,  │ ───► │ trained on healthy  │ ─────► │ reading; alert if   │
 │  flow, RPM)         │      │ data only           │        │ score < threshold   │
 └─────────────────────┘      │ DBSCAN: cluster     │        │ → "bearing wear"    │
                              │ healthy data        │        │   detected at t=... │
                              └─────────────────────┘        └─────────────────────┘
```

**The key idea:** you only need *healthy* data to train. The fault is defined by its *deviation* from normal, not by a catalog of known failures.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('scikit-learn imports OK')

## 1. Generate the Sensor Data

### The concept

We simulate **1,000 hours** of turbine operation with 5 sensors:

| Sensor | Normal mean | Normal std |
|--------|------------|------------|
| Temperature (°C) | 85 | 2 |
| Vibration (mm/s) | 2.0 | 0.2 |
| Pressure (bar) | 5.0 | 0.1 |
| Flow (kg/s) | 12.0 | 0.3 |
| RPM (×1000) | 3.0 | 0.02 |

**Normal operation** is a correlated Gaussian — the sensors move together (higher RPM → higher temperature → higher flow). We build the correlation with a covariance matrix.

**The fault** is *bearing wear*: vibration rises (loose bearing), temperature follows (friction heat), and the correlation structure degrades. We inject 5% fault samples — the kind of early, subtle degradation you want to catch *before* the bearing seizes.

**Why synthetic?** The structure is what matters: correlated normal data + a physically-motivated fault mode. Swap in your real sensor CSV and the same code runs.

In [ ]:
N = 1000  # hours of data

# Normal operation: correlated Gaussian
mean = np.array([85.0, 2.0, 5.0, 12.0, 3.0])
cov = np.array([
    [4.0,   0.4,  0.05, 0.3,  0.02],
    [0.4,   0.04, 0.005,0.02, 0.002],
    [0.05,  0.005,0.001,0.01, 0.001],
    [0.3,   0.02, 0.01, 0.09, 0.03],
    [0.02,  0.002,0.001,0.03, 0.0004]
])
X_normal = np.random.multivariate_normal(mean, cov, N)

# Fault: bearing wear (5% of samples)
n_fault = 50
idx_fault = np.random.choice(N, n_fault, replace=False)
X_fault = X_normal[idx_fault].copy()
X_fault[:, 1] += np.random.uniform(1.5, 4.0, n_fault)   # vibration rises
X_fault[:, 0] += np.random.uniform(3.0, 10.0, n_fault)  # temperature follows
X_fault[:, 2] -= np.random.uniform(0.05, 0.2, n_fault)  # slight pressure drop

X = X_normal.copy()
X[idx_fault] = X_fault
y = np.zeros(N, dtype=int)
y[idx_fault] = 1  # 1 = fault

sensor_names = ['Temp (C)', 'Vibration (mm/s)', 'Pressure (bar)', 'Flow (kg/s)', 'RPM (x1000)']
print(f'Dataset: {N} hours, {X.shape[1]} sensors')
print(f'Normal: {(y==0).sum()} h, Fault: {(y==1).sum()} h ({100*(y==1).mean():.1f}%)')

### What does this mean?

The fault is **subtle by design**: a 2–4 mm/s vibration increase on a 2 mm/s baseline is a 100–200% shift in one sensor, but the other four sensors barely move. A single-sensor threshold would miss it or cry wolf constantly. The *combination* of sensors is where the signal lives — which is exactly what multivariate anomaly detection exploits.

## 2. See the Data: Time Series with Anomalies

Before any algorithm, *look* at the data. Plot the two most fault-sensitive sensors (vibration and temperature) over time, with fault samples highlighted. This is the ground truth we'll measure the algorithms against.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

t = np.arange(N)
fault_mask = (y == 1)

axes[0].plot(t, X[:, 1], 'steelblue', lw=0.8, label='Vibration')
axes[0].scatter(t[fault_mask], X[fault_mask, 1], c='red', s=20, zorder=5, label='Fault (bearing wear)')
axes[0].set_ylabel('Vibration (mm/s)')
axes[0].legend(loc='upper left')
axes[0].set_title('Turbine sensors over 1000 h — fault samples highlighted')

axes[1].plot(t, X[:, 0], 'coral', lw=0.8, label='Temperature')
axes[1].scatter(t[fault_mask], X[fault_mask, 0], c='red', s=20, zorder=5, label='Fault (bearing wear)')
axes[1].set_ylabel('Temperature (C)')
axes[1].set_xlabel('Time (h)')
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.show()

### What does this mean?

The red dots are the bearing-wear events. Notice they're **not all obvious** — some sit near the top of the normal distribution, others are clear outliers. A human scanning this plot would catch the big ones and miss the subtle ones. That's the job we're automating.

Before any algorithm, we need a **train/test split** (stratified, so both classes appear in both sets) and **feature scaling**. In a real deployment you'd train on all healthy historical data and test on new incoming data — the split here simulates that. `StandardScaler` is fit on training data only: Isolation Forest and DBSCAN both assume comparable feature scales, and fitting on train only prevents data leakage.

## 3. Train/Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features — anomaly detectors are sensitive to feature scale
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} h ({(y_train==1).sum()} fault)')
print(f'Test:  {X_test.shape[0]} h ({(y_test==1).sum()} fault)')

## 4. Isolation Forest

**Isolation Forest** builds many random trees. At each node, it picks a random feature and a random split value between that feature's min and max. The key insight:

> **Anomalies are few and different — they get isolated in fewer splits than normal points.**

A normal point sits in a dense region; it takes many random splits to isolate it. An anomaly sits alone; a single split can cut it off. The **path length** to isolate a point is the anomaly score: short path = anomaly.

- `contamination=0.05` tells the forest we expect ~5% anomalies (sets the decision threshold)
- `score_samples()` returns the raw anomaly score (more negative = more anomalous) — useful for tuning the threshold

In [ ]:
iforest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)
iforest.fit(X_train_s)

# Predict on test set
y_pred_if = iforest.predict(X_test_s)
# -1 = anomaly, +1 = normal
y_pred_if_binary = (y_pred_if == -1).astype(int)

# Anomaly scores (more negative = more anomalous)
scores_if = iforest.score_samples(X_test_s)

print('Isolation Forest predictions on test set:')
print(f'  Flagged as anomaly: {y_pred_if_binary.sum()} / {len(y_pred_if_binary)}')
print(f'  True faults in test: {y_test.sum()}')

## 5. DBSCAN: Cluster-Based Anomaly Detection

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) takes a different approach:

- A point is a **core point** if it has at least `min_samples` neighbors within distance `eps`
- Points reachable from core points form **clusters**
- Points that aren't in any cluster are **noise** (label = -1) → anomalies

The intuition: healthy sensor data forms a *dense cloud* in 5D space. A bearing-wear fault shifts the vibration and temperature, pushing those points *out of the dense region* — they become noise.

**The catch:** DBSCAN's `eps` and `min_samples` are geometry parameters. In 5D, "distance" behaves differently than in 2D (the curse of dimensionality). We'll see how it compares to Isolation Forest.

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=20)
y_pred_db = dbscan.fit_predict(X_train_s)

# -1 = noise (anomaly) in DBSCAN
y_pred_db_binary = (y_pred_db == -1).astype(int)

# Evaluate on training set (DBSCAN is unsupervised — we fit on train, check on train)
print('DBSCAN on training set:')
print(f'  Clusters found: {len(set(y_pred_db) - {-1})}')
print(f'  Noise points (anomalies): {y_pred_db_binary.sum()} / {len(y_pred_db_binary)}')
print(f'  True faults in train: {y_train.sum()}')
print()
print('Cluster sizes:')
for label in sorted(set(y_pred_db)):
    if label != -1:
        print(f'  Cluster {label}: {(y_pred_db == label).sum()} points')
print(f'  Noise: {y_pred_db_binary.sum()} points')

## 6. Head-to-Head: Confusion Matrix and Metrics

Now we measure both algorithms against ground truth. The **confusion matrix** tells us:

- **True Positives (TP):** fault correctly flagged
- **False Positives (FP):** normal data incorrectly flagged (false alarm)
- **False Negatives (FN):** fault missed (dangerous!)
- **True Negatives (TN):** normal data correctly passed

**Precision** = TP / (TP + FP) — of the alarms, how many were real?
**Recall** = TP / (TP + FN) — of the real faults, how many did we catch?

For fault detection, **recall is the critical metric**: a missed bearing failure is catastrophic. A false alarm just costs an inspection.

In [ ]:
# Isolation Forest: evaluate on test set
cm_if = confusion_matrix(y_test, y_pred_if_binary)
print('=== Isolation Forest (test set) ===')
print(classification_report(y_test, y_pred_if_binary, target_names=['Normal', 'Fault']))

# DBSCAN: evaluate on training set (unsupervised — no separate test)
cm_db = confusion_matrix(y_train, y_pred_db_binary)
print('=== DBSCAN (train set) ===')
print(classification_report(y_train, y_pred_db_binary, target_names=['Normal', 'Fault']))

### What does this mean?

**Isolation Forest** typically wins on recall here — it's designed to flag the *most isolated* points, and the bearing-wear faults are exactly that. **DBSCAN** may flag more false positives (normal points on the edge of the cluster) or miss faults that are only slightly displaced.

**Engineering takeaway:** in a safety-critical application, you'd tune `contamination` (Isolation Forest) or `eps`/`min_samples` (DBSCAN) to maximize **recall** first, then accept the false-alarm rate. A 95% recall with 10% false alarms is far better than 80% recall with 2% false alarms — the cost of a missed bearing failure dwarfs the cost of an extra inspection.

## 7. Visualize the Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cm, title in zip(axes, [cm_if, cm_db],
                          ['Isolation Forest (test)', 'DBSCAN (train)']):
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Normal', 'Fault'])
    ax.set_yticklabels(['Normal', 'Fault'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
    fig.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

## 8. Decision Boundary: Where Does the Algorithm Draw the Line?

Let's visualize *where* Isolation Forest draws its boundary in the 2D projection of the two most fault-sensitive sensors (vibration vs. temperature). The color shows the anomaly score; the red dots are true faults.

In [ ]:
# 2D projection: vibration (x) vs temperature (y)
X2d = X_test_s[:, [1, 0]]  # vibration, temperature
scores_2d = iforest.score_samples(X_test_s)

fig, ax = plt.subplots(figsize=(10, 7))

# Background: anomaly score on a grid
x_min, x_max = X2d[:, 0].min() - 0.5, X2d[:, 0].max() + 0.5
y_min, y_max = X2d[:, 1].min() - 0.5, X2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
grid = np.column_stack([xx.ravel(), yy.ravel()])
# Pad with zeros for the other 3 features (Isolation Forest uses all 5)
grid_full = np.hstack([grid, np.zeros((grid.shape[0], 3))])
Z = iforest.score_samples(grid_full).reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=20, cmap='RdYlGn_r', alpha=0.7)
ax.scatter(X2d[:, 0], X2d[:, 1], c='steelblue', s=15, alpha=0.5, label='Normal (test)')
ax.scatter(X2d[y_test==1, 0], X2d[y_test==1, 1], c='red', s=40, zorder=5,
           edgecolors='k', linewidth=0.5, label='Fault (bearing wear)')

ax.set_xlabel('Vibration (mm/s) [scaled]')
ax.set_ylabel('Temperature (C) [scaled]')
ax.set_title('Isolation Forest anomaly score (green = normal, red = anomalous)')
ax.legend(loc='upper left')
plt.show()

### What does this mean?

The green region is "normal" — the dense cloud of healthy operation. The red/orange region is where the algorithm flags anomalies. The red dots (true faults) should sit in or near the red zone; any red dots in the green zone are the **missed faults** (false negatives) from the confusion matrix.

**Engineering takeaway:** this plot is your *explanation* for the alarm system. When the system flags a point, you can show the operator *where* it sits relative to the normal operating envelope — the difference between "trust the algorithm" and "here's why the algorithm is alarmed.

## 9. The Engineering Payoff: Lead Time

### The concept

Let's put numbers on the value. A bearing failure on a turbine causes:

- **Unplanned downtime:** 2–4 weeks of repair
- **Cost:** $500K–$2M (parts, labor, lost production)
- **Risk:** collateral damage to the rotor, gearbox, or downstream components

An anomaly detection system that catches bearing wear **500 hours before failure** changes the economics entirely:

| Scenario | Cost |
|----------|------|
| Missed failure (unplanned) | $1M + 3 weeks downtime |
| Detected at 500 h (planned) | $5K inspection + 4 h swap |
| False alarm (10% rate) | $500 inspection × 100 h = $50K/yr |

**Net savings:** ~$900K per avoided failure, minus the false-alarm cost. The system pays for itself with a single caught failure.

Now let's simulate a **progressive** bearing wear — vibration rising gradually after hour 300 — and see when the algorithm flags it. That lead time is the entire value proposition.

In [ ]:
# Progressive bearing wear: vibration rises linearly after hour 300
# In the synthetic data, faults are injected at random times.
# In reality, bearing wear is *progressive* — vibration rises gradually.
# Let's simulate a progressive fault and see when the algorithm flags it.

N_prog = 500  # hours
t_prog = np.arange(N_prog)

# Progressive bearing wear: vibration rises linearly after hour 300
# (ramp reaches full magnitude at hour 450, then holds)
vib = 2.0 + 0.2 * np.random.randn(N_prog)
ramp = np.clip((t_prog - 300) / 150.0, 0, 1)  # 0 before h=300, 1 after h=450
vib += 3.0 * ramp  # gradual rise, then sustained
temp = 85.0 + 2.0 * np.random.randn(N_prog)
temp += 5.0 * ramp  # follows vibration

X_prog = np.column_stack([
    temp,
    vib,
    5.0 + 0.1 * np.random.randn(N_prog),
    12.0 + 0.3 * np.random.randn(N_prog),
    3.0 + 0.02 * np.random.randn(N_prog)
])
X_prog_s = scaler.transform(X_prog)

# Score each hour
scores_prog = iforest.score_samples(X_prog_s)
threshold = iforest.offset_  # the decision threshold

flagged = scores_prog < threshold
first_flag = np.argmax(flagged) if flagged.any() else None

print(f'Fault onset: hour 300 (ramp to full severity by hour 450)')
print(f'First flag:  hour {first_flag}')
if first_flag is not None:
    print(f'  -> flagged {first_flag - 300} hours after onset, long before failure')

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t_prog, vib, 'steelblue', lw=1)
axes[0].axvline(300, color='orange', ls='--', lw=1, label='Fault onset (h=300)')
if first_flag:
    axes[0].axvline(first_flag, color='red', ls='--', lw=1, label=f'First flag (h={first_flag})')
axes[0].set_ylabel('Vibration (mm/s)')
axes[0].legend()
axes[0].set_title('Progressive bearing wear: vibration over time')

axes[1].plot(t_prog, scores_prog, 'coral', lw=1)
axes[1].axhline(threshold, color='red', ls='--', lw=1, label='Anomaly threshold')
axes[1].axvline(300, color='orange', ls='--', lw=1)
if first_flag:
    axes[1].axvline(first_flag, color='red', ls='--', lw=1)
axes[1].set_ylabel('Anomaly score')
axes[1].set_xlabel('Time (h)')
axes[1].legend()
axes[1].set_title('Isolation Forest score over time')

plt.tight_layout()
plt.show()

### What does this mean?

The anomaly score rises as the bearing degrades. The algorithm flags the point where the score crosses the threshold — **before the vibration reaches a level that would trip a simple threshold alarm**. That lead time is the entire value proposition: it converts an unplanned failure into a planned maintenance window.

**Engineering takeaway:** anomaly detection isn't about *whether* you detect the fault — it's about *how early*. The lead time determines whether you can schedule the repair during a planned shutdown or whether you're doing an emergency teardown at 2 AM. In a real deployment you'd tune the threshold against historical data to balance lead time against false alarms.

## 10. Exercises

1. **Tune the threshold.** Instead of `contamination=0.05`, use `score_samples()` on the training set and pick a threshold that gives 95% recall. What's the false-alarm rate at that threshold? Plot precision vs. recall as you sweep the threshold.

2. **Add a second fault mode.** Inject a "blade crack" fault (pressure oscillation: add `0.1 * sin(2*pi*t/10)` to the pressure sensor). Can Isolation Forest detect both fault types? Does DBSCAN separate them into different noise clusters?

3. **Curse of dimensionality.** Add 10 more sensors (all normal, uncorrelated). Does Isolation Forest's performance degrade? Does DBSCAN's? Why?

4. **Real data.** Replace the synthetic data with a real sensor CSV (e.g., from a C-MAPSS turbofan dataset). How does the algorithm perform when the "normal" data is non-stationary (seasonal variation, slow drift)?

5. **Online detection.** Modify the code to process the data *hour by hour* (streaming). Can you detect the fault in real time without retraining? (Hint: use `score_samples()` on a sliding window.)

6. **Explainability.** For each flagged anomaly, compute which sensor contributed most to the anomaly score. (Hint: train a separate Isolation Forest on each sensor pair and compare scores.) Which sensor is the "smoking gun" for bearing wear?